### 00. Utils

In [ ]:
from random import sample,randrange
from typing import List
from pysat.solvers import Glucose3
import numpy as np

def use_all_variables(variables:List[int],matriz:List[list])->bool:
    """Garante que as claúsulas usem todos os elementos."""
    for variable in variables:
        contains = False
        for clause in matriz:
            if variable in clause or -variable in clause:
                contains = True
                break
        if not contains:
            clause = sample(matriz,1)[0]
            clause[randrange(0,len(clause))] =  sample([-1,1],1)[0] * variable
            return use_all_variables(variables,matriz)
    return True
                
def generate_sat(number_variables:int,number_clauses:int,type_sat:int)-> List[list]:
    """
    Gera uma fórmula SAT (Satisfiability Problem) com base no número de variáveis, 
    número de cláusulas e tipo de SAT especificado.

    Args:
        number_variables (int): Número de variáveis proposicionais únicas.
        number_clauses (int): Número de cláusulas a serem geradas.
        type_sat (str): Tipo de problema SAT (exemplos: 3-SAT, 5-SAT).

    Returns:
        List[List[int]]: Lista de cláusulas, onde cada cláusula é uma lista de literais inteiros.
                        Literais positivos indicam variáveis afirmadas, e negativos, suas negações.
    
    Example:
        >>> generate_sat(3, 2,3)
        [[1, -2, 3], [-1, 2, -3]]
    """
    if number_variables > number_clauses * type_sat:
        raise ValueError("Número de variáveis insuficiente para satisfazer as cláusulas.")
    
    variables = list(range(1,number_variables+1))

    matriz = []
    
    while len(matriz) < number_clauses:
        clause = []
        
        while len(clause) < type_sat:
            value = sample(variables,1)[0]
            
            value = value * sample([-1,1],1)[0]
            
            value not in clause and (-value) not in clause and clause.append(value)
        
        clause not in matriz and matriz.append(clause)
        
    use_all_variables(variables,matriz)
        
    return matriz

def is_satisfiable(instance):
    """Verifica se uma instância SAT é satisfazível"""
    solver = Glucose3(use_timer=True)
    for clause in instance:
        solver.add_clause(clause)
        
    return solver.solve(), solver.time()

### 02. Testes 

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import time


def calc_and_plot(alpha_values,type_sat:int):
    start_time = time.time()  # Captura o tempo inicial
    number_variables = [20,30,40,50,60]
    colors = ["#D32F2F", "#FFA726", "#A020F0", "#00838F", "#FF6F00"]
    num_instances = 30
    results = {number: {} for number in number_variables}

    for number in number_variables:
        print(f"Calculando para {number} variáveis.")
        probabilities = []
        avg_times = []
        alphas = []
        
        for alpha in alpha_values:
            satisfiable_count = 0
            total_time = 0
            for _ in range(num_instances):
                number_clauses = int(alpha * number)
                matriz = generate_sat(number_variables=number,number_clauses=number_clauses,type_sat=type_sat)
                satisfiable, time_used = is_satisfiable(matriz)
                
                if satisfiable:
                    satisfiable_count += 1
                    
                total_time += time_used
        
            probability = satisfiable_count / num_instances
            avg_time = total_time / num_instances

            probabilities.append(probability)
            avg_times.append(avg_time)
            alphas.append(alpha)
        results[number]['alphas'] = alphas
        results[number]['probabilities'] = probabilities
        results[number]['avg_times'] = avg_times

    plt.figure(figsize=(10,5))
    for index, key in enumerate(results):
        alphas = results[key]['alphas']
        probabilities = results[key]['probabilities']
        
        plt.plot(alphas,probabilities,label=f"n={key}",color=colors[index])
        
    plt.legend()
    plt.xticks(np.arange(alpha_values[0], alpha_values[-1]+1, round(alpha_values[-1]/20,2))) 
    plt.yticks(np.arange(0, 1.2,0.2)) 
    plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: f"{int(x*100)}%"))
    plt.xlabel("α")
    plt.ylabel("Probabilidade de satisfazibilidade")
    plt.title(f"Probabilidade em função de alpha para {type_sat}-SAT", fontsize=16, weight='bold', pad=20, loc='center')

    plt.figure(figsize=(10,5))
    for index, key in enumerate(results):
        alphas = results[key]['alphas']
        avg_times = results[key]['avg_times']
        
        plt.plot(alphas,avg_times,label=f"n={key}",color=colors[index])
        
    plt.legend()
    plt.xticks(np.arange(alpha_values[0], alpha_values[-1]+1,round(alpha_values[-1]/20,2)))
    plt.xlabel("α")
    plt.ylabel("Tempo gasto(s)")
    plt.title(f"Tempo em função de alpha para {type_sat}-SAT", fontsize=16, weight='bold', pad=20, loc='center')       
    end_time = time.time()  # Captura o tempo final
    execution_time = end_time - start_time  # Calcula o tempo decorrido

    print(f"Tempo de execução para {type_sat}-SAT: {execution_time:.6f} segundos")

alpha_values = [round(x, 2) for x in np.arange(1,10.1,0.1)]     
calc_and_plot(alpha_values,3)
alpha_values = [round(x, 2) for x in np.arange(1, 30.1, 0.1)]   
calc_and_plot(alpha_values,5)